In [ ]:
import plotly.graph_objects as go
import scipy.constants as cst
import plotly.express as px
import numpy as np
import inspect

# **Paramétrages**

In [ ]:
def parametrage_tore(phi, theta, R=2, r=1):
    X = (R + r * np.cos(theta)) * np.cos(phi)
    Y = (R + r * np.cos(theta)) * np.sin(phi)
    Z = r * np.sin(theta)
    return X,Y,Z

def parametrage_sphere(phi, theta, R=1):              
    X = R * np.sin(theta) * np.cos(phi)
    Y = R * np.sin(theta) * np.sin(phi)
    Z = R * np.cos(theta)
    return X,Y,Z

def parametrage_cylindre(theta, z, R=1):           
    X = R * np.cos(theta)
    Y = R * np.sin(theta)
    return X,Y, z

def parametrage_disque(rayon, theta, z=0):
    X = rayon * np.cos(theta) 
    Y = rayon * np.sin(theta)  
    Z = np.full_like(theta, z)
    return X,Y,Z

def parametrage_plan(longueur,largeur,z=0) : 
    X = longueur
    Y = largeur
    Z = np.full_like(X,z)
    return X,Y,Z

## **Niveau 1**

In [ ]:
class Surface :
    
    def __init__(self, parametrage : callable, domaine_u : tuple[float,float,bool], domaine_v : tuple[float,float,bool], densite : int, *args) -> "Surface":
        """
        Entrée : 
        
            - parametrage: fonction de la forme f(u,v, *args) qui renvoie un tuple (x,y,z) pour les coordonnées de la surface.
            
            - domaine_u: tuple (min_u, max_u, ouvert_u) où min_u et max_u sont les bornes du domaine de u et ouvert_u est un booléen
            indiquant si la borne supérieure est ouverte ou fermée.
            
            - domaine_v: tuple (min_v, max_v, ouvert_v) où min_v et max_v sont les bornes du domaine de v et ouvert_v est un booléen
            indiquant si la borne supérieure est ouverte ou fermée.
            
            - densite: nombre de points dans chaque direction (u et v) pour le maillage de la surface.
            
            - args: arguments supplémentaires à passer à la fonction de paramétrage.
        
        Object:
         
            - Crée une surface paramétrée par la fonction de paramétrage donnée sur le domaine spécifié.
            
            - Il possède des caractértiques intrasèsques:
            
                - parametrage: la fonction de paramétrage.
                
                - domaine_u: le domaine de u.
                
                - domaine_v: le domaine de v.
                
                - densite: le nombre de points dans chaque direction.
                
                - args: les arguments supplémentaires pour la fonction de paramétrage.
                
                - u: les valeurs de u sur le maillage.
                
                - v: les valeurs de v sur le maillage.
                
                - U: la grille de valeurs de u.
                
                - V: la grille de valeurs de v.
                
                - X: les coordonnées x de la surface.
                
                - Y: les coordonnées y de la surface.
                
                - Z: les coordonnées z de la surface.                    
        """
        
        ouvert_u = domaine_u[2] if len(domaine_u) == 3 else False 
        ouvert_v = domaine_v[2] if len(domaine_v) == 3 else False 
        
        u = np.linspace(domaine_u[0],domaine_u[1],densite, endpoint=not(ouvert_u))
        v = np.linspace(domaine_v[0],domaine_v[1],densite, endpoint=not(ouvert_v))
        U, V = np.meshgrid(u,v)
        
        X, Y, Z = parametrage(U,V, *args)
        self.parametrage = parametrage
        self.domaine_u = domaine_u
        self.domaine_v = domaine_v
        self.densite = densite
        self.args = args
        self.u = u
        self.v = v
        self.U = U
        self.V = V
        self.X : np.ndarray = X
        self.Y : np.ndarray = Y
        self.Z : np.ndarray = Z
        
    def afficher_definition(self) -> None:
        """"
        Affiche la définition de la surface.
        """
        try:
            source = inspect.getsource(self.parametrage)
            print("Ligne d'instance :", source)
        except TypeError:
            print("Impossible d'afficher la source.")
    
    def calculer_dS(self) -> np.ndarray :
        """
        Entrée :
        
            - None.
        
        Sortie :
            
            - dS: tableau 1D contenant les éléments de surface pour chaque point de la grille.
        """
        du = (self.domaine_u[1] - self.domaine_u[0]) / self.densite
        dv = (self.domaine_v[1] - self.domaine_v[0]) / self.densite
        
        U = self.U
        V = self.V
        parametres = self.args
        parametrisation = self.parametrage

        X,Y,Z = parametrisation(U, V, *parametres)
        X_du, Y_du, Z_du = parametrisation(U + du, V, *parametres)
        X_dv, Y_dv, Z_dv = parametrisation(U, V + dv, *parametres)
        
        r = np.stack([X,Y,Z], axis=-1)
        ru = (np.stack([X_du, Y_du, Z_du], axis=-1) - r) / du
        rv = (np.stack([X_dv, Y_dv, Z_dv], axis=-1) - r) / dv

        normal = np.cross(ru, rv, axis=-1)  
        dS : np.ndarray = np.linalg.norm(normal, axis=-1)
        
        return dS.flatten() * du * dv
    
    def calculer_aire(self) -> float:
        """
        Entrée :
        
            - None.
            
        Sortie :
            
            - aire: aire totale de la surface.
        """
        return np.sum(self.calculer_dS())

## **Niveau 2**

In [ ]:
class Maillage : 
    """
    Entrées :
        
        - X : C'est un np.ndarray qui représente toutes les valeurs de x prises par le maillage.
       
         - Y : C'est un np.ndarray qui représente toutes les valeurs de y prises par le maillage.
        
        - Z : C'est un np.ndarray qui représente toutes les valeurs de z prises par le maillage.
        
        - surface : C'est une liste contenant toutes les surfaces qui composent le maillage.
        
        NB : Les matrices X,Y,Z doivent avoir les mêmes dimensions.
    
    Objet :
        
        - Il possède des caractéristiques intrinsèques :
        
            - X : est la version flatten de X pris en entrée. Renvoie la matrice.
            
            - Y : est la version flatten de Y pris en entrée. Renvoie la matrice.
            
            - Z : est la version flatten de Z pris en entrée. Renvoie la matrice.
            
            - XYZ : renvoie le tuple (X,Y,Z) où les matrices sont flatten.
            
            - nb_points : Renvoie le nombre total de points contenus dans le maillage.
    """
    def __init__(self,X : np.ndarray = None, Y : np.ndarray = None, Z : np.ndarray = None, surface : list[Surface] = None) -> "Maillage" :       
        
        if surface is not None:
            for elem in surface:
                if not isinstance(elem, Surface) : 
                    raise TypeError('La surface doit être de type Surface.')
            self.surface = surface
            if X is None or Y is None or Z is None:
                if len(surface) != 1:
                    raise ValueError('Il doit y avoir une seule surface.')
                self.X = surface[0].X.flatten()
                self.Y = surface[0].Y.flatten()
                self.Z = surface[0].Z.flatten()
                self.XYZ = np.array([self.X, self.Y, self.Z])
                self.nb_points = surface[0].densite ** 2
            else:
                if not((size := X.size) == Y.size == Z.size) : 
                    raise ValueError('Les 3 matrices doivent avoir les mêmes dimensions.')
                self.X = X.flatten()
                self.Y = Y.flatten()
                self.Z = Z.flatten()
                self.XYZ = np.array([self.X, self.Y, self.Z])
                self.nb_points = size
        else:
            if X is None or Y is None or Z is None:
                raise ValueError('X, Y et Z doivent être fournis.')
            if not((size := X.size) == Y.size == Z.size) : 
                raise ValueError('Les 3 matrices doivent avoir les mêmes dimensions.')
            self.surface = None
            self.X = X.flatten()
            self.Y = Y.flatten()
            self.Z = Z.flatten()
            self.XYZ = np.array([self.X, self.Y, self.Z])
            self.nb_points = size
    
    def __add__(self,maillage : "Maillage") -> "Maillage" :
        """
        Entrée :
        
            - maillage : C'est un autre maillage à additionner au maillage courant.
            
        Sortie :
        
            - Renvoie un nouveau maillage qui est la somme des deux maillages. 
            
        NB : La somme est définie par la concaténation des matrices X,Y,Z deux à deux des deux maillages.
        """ 
        X,Y,Z = np.concatenate((self.X,maillage.X)),np.concatenate((self.Y,maillage.Y)),np.concatenate((self.Z,maillage.Z))
        
        return  Maillage(X = X, Y = Y, Z = Z, surface = [*self.surface, *maillage.surface])
    
    def __rmul__(self,scalaire) :
        """
        Entrée :
        
            - scalaire : C'est un float qui va multiplier le maillage courant.
            
        Sortie :
        
            - Renvoie un nouveau maillage qui est le maillage multiplié par le scalaire.
            
        NB : La multiplication est définie par la multiplication des matrices X,Y,Z du maillage par le scalaire.
        """
        return Maillage(X = scalaire*self.X, Y = scalaire*self.Y, Z = scalaire*self.Z, surface = self.surface) 
    
    def __getitem__(self,index : int) -> "Maillage" : 
        """
        Entrée :
            
                - index : C'est un entier positif qui représente l'indice de l'élément voulu.
                
        Sortie :
            
                - Renvoie un tuple contenant les coordonnées (X,Y,Z) de l'élément voulu.
                
        NB : L'indice doit être positif et inférieur au nombre total de points du maillage - 1.
        """
        return self.X[index],self.Y[index],self.Z[index]
    
    def __eq__(self, maillage : "Maillage"):
        """
        Entrée :
        
            - maillage : C'est un autre maillage à comparer au maillage courant.
            
        Sortie :
            
                - Renvoie True si les deux maillages sont égaux, False sinon.
                
        NB : Deux maillages sont égaux si et seulement si leurs trois matrices X,Y,Z sont égales selon la fonction numpy. array_equal.
        """
        return np.array_equal(self.X, maillage.X) and \
               np.array_equal(self.Y, maillage.Y) and \
               np.array_equal(self.Z, maillage.Z)

    def translate(self, dl : tuple[float]) -> "Maillage":
        """
        Entrée :
        
            - dl : C'est un vecteur (1x3) (dx,dy,dz) qui contient les informations de combien la distribution va être
            translatée selon les 3 axes cartésiens. Mettre 0 pour les axes qui ne bougent pas.
        
        Sorties :

            - Renvoie un nouveau maillage qui tient compte de la translation.
        """
        if len(dl) != 3:
            raise ValueError('Trop ou pas assez de translations ont été données.')

        return Maillage(X = self.X + dl[0], Y = self.Y + dl[1], Z = self.Z + dl[2], surface = self.surface)    
    
    def rotate(self, axes : list[str], angles : list[float]) -> "Maillage" : 
        """
        Entrées :
            
            - axes : C'est une liste de string. Elles sont soit « x », « y » ou « z » et représentent les axes de rotation du
            maillage.
        
            - angles : C'est une liste de float. Ils représentent, dans l'ordre des axes nommés dans la liste « axes », les
            angles de rotation autours des axes.
        
        Sortie :

            - Renvoie le nouveau maillage avec les rotations effectives.
        """
        if len(axes) != len(angles):
            raise ValueError("Autant d'angles que d'axes doivent être donnés.")
        if len(axes) > 3 or len(angles) > 3:
            raise ValueError("Trop d'angles et/ou axes ont été donnés.")
        for elem in axes:
            if elem.strip().lower() not in ["x","y","z"]:
                raise ValueError("Les axes donnés ne sont pas bons.")

        Rx = lambda angle : np.array([[1,0,0],[0,np.cos(angle),-np.sin(angle)],[0,np.sin(angle),np.cos(angle)]])
        Ry = lambda angle : np.array([[np.cos(angle),0,np.sin(angle)],[0,1,0],[-np.sin(angle),0,np.cos(angle)]])
        Rz = lambda angle : np.array([[np.cos(angle),-np.sin(angle),0],[np.sin(angle),np.cos(angle),0],[0,0,1]])
        product = ''
    
        for axe,angle in zip(axes,angles) : 
            product = f'R{axe.strip().lower()}({angle}) @' + product
        
        return Maillage(*eval(f'{product} self.XYZ'), surface = self.surface) 
    
    def calculer_aire(self) -> float :
        """
        Entrée : 
        
            - None.
        
        Sortie : 
        
            - aire : C'est un float qui représente l'aire totale du maillage.
        """
        aire = 0
        for surface in self.surface:
            aire += surface.calculer_aire()
        return aire
    
    def afficher_3D_plotly(self) -> None: 
        """
        Affiche le maillage en 3D à l'aide de Plotly.
        """
        fig = go.Figure(data=[go.Scatter3d(x=self.X, y=self.Y, z=self.Z, mode='markers', marker=dict(size=2))])
        fig.update_layout(scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))
        fig.show()
        return None
    
def sum(*M : list[Maillage]) -> Maillage : 
    """
    Entrée :
    
        - M : C'est une liste de maillages à additionner.
        NB : La somme est définie par la concaténation des matrices X,Y,Z deux à deux des maillages.
        
    Sortie :
    
        - Renvoie un nouveau maillage qui est la somme des maillages.
    """
    m = M[0]
    for i in range(1, len(M)) : 
        m = m + M[i]
    return m


## **Niveau 3**

In [ ]:
def creer_distribution(parametrage : callable, domaine_u : tuple[float], domaine_v : tuple[float], densite : int, *args) -> Maillage :
    """
    Entrée :
    
        - parametrage : C'est une fonction qui prend en entrée deux matrices (U,V) et renvoie trois matrices (X,Y,Z).
        
        - domaine_u : C'est un tuple de float qui représente le domaine définition de la variable u.
        
        - domaine_v : C'est un tuple de float qui représente le domaine définition de la variable v.
        
        - densite_u : C'est un entier positif qui représente le nombre de point selon u.
        
        - densite_v : C'est un entier positif qui représente le nombre de point selon v. 
        
    Sortie :
    
        - Renvoie une instance de la classe Maillage contenant la surface définie par le paramétrage et les domaines donnés.
    """
    return Maillage(surface=[Surface(parametrage, domaine_u, domaine_v, densite, *args)])

## **Niveau 4**

In [ ]:
def calcule_charge_dS(surface : Surface, distribution_charge):
    """
    Entrée :
    
        - surface : C'est une instance de la classe Surface.
        
        - distribution_charge : C'est un tuple contenant la charge totale et une fonction f qui calcule la distribution de charge.
        NB : Si f est None, la charge est uniformément répartie sur la surface.
        
    Sortie :
        
        - Renvoie un tableau contenant la distribution de charge sur la surface pour chaque point.
    """
    dS = surface.calculer_dS()
    charge_totale, f = distribution_charge
    
    if f is None:
        sigma = charge_totale / np.sum(dS)
        return sigma * dS
    else: 
        poids = f(surface.U, surface.V).ravel()
        sigma = charge_totale / np.sum(poids * dS)
        return sigma * poids * dS

In [ ]:
def calcule_charge(maillage : Maillage, distribution_charge):
    """
    Entrée :
    
        - maillage : C'est une instance de la classe Maillage.
        
        - distribution_charge : C'est un tuple contenant la charge totale et une fonction f qui calcule la distribution de charge.
        NB : Si f est None, la charge est uniformément répartie sur le maillage.
        
    Sortie :
        
        - Renvoie un tableau contenant la distribution de charge sur le maillage pour chaque point.
    """
    dS_tot = maillage.calculer_aire()
    charge_totale, f = distribution_charge
    
    dS = maillage.surface[0].calculer_dS()
    charge = calcule_charge_dS(maillage.surface[0], (charge_totale*np.sum(dS)/dS_tot, f))
    if f is not None:
        poids = f(maillage.surface[0].U, maillage.surface[0].V)
    
    for surface in maillage.surface[1:] : 
        dS = np.concatenate([dS,surface.calculer_dS()]) 
        charge = np.concatenate([charge,calcule_charge_dS(surface, (charge_totale*np.sum(surface.calculer_dS())/dS_tot, f))])
        if f is not None:
            poids = np.concatenate([poids,f(surface.U,surface.V)])
    
    if f is None:
        sigma = charge_totale / np.sum(dS)
        return sigma * dS
    else: 
        sigma = charge_totale / np.sum(poids.ravel() * dS)
        return sigma * poids.ravel() * dS

## **Niveau 5**

In [ ]:
def calcule_champ(distribution: Maillage, maillage: Maillage, distribution_charge, filtre : bool = False) -> np.ndarray :
    """
    Entrée :
    
        - distribution : C'est une instance de la classe Maillage contenant la distribution de charge.
        
        - maillage : C'est une instance de la classe Maillage contenant le maillage sur lequel on veut calculer le champ électrique.
        
        - distribution_charge : C'est un tuple contenant la charge totale et une fonction f qui calcule la distribution de charge.
        NB : Si f est None, la charge est uniformément répartie sur la surface.
        
    Sortie :
        
        - Renvoie un tableau 2D contenant le champ électrique calculé sur le maillage donné. Les lignes représentent les valeurs du champ
        en x, y, z et les colonnes représentent les points du maillage.
    """
    k = 1 / (4 * np.pi * cst.epsilon_0)
    val_charge_tot, fonction_charge = distribution_charge
        
    charge = calcule_charge(distribution, (val_charge_tot, fonction_charge))
    
    r = (-1*distribution.XYZ.T[None, :, :] + maillage.XYZ.T[:, None, :])
    norm = np.linalg.norm(r,axis=2)
    if filtre:
        norm = np.where(norm >= 1e-6, norm, 1e30)
        
    E = np.sum(k*r* charge[None,:,None] / norm[:,:,None]**3,axis=1)

    return E


In [ ]:
def calcule_potentiel(distribution: Maillage, maillage: Maillage, distribution_charge, filtre : bool = False) -> np.ndarray :
    """
    Entrée :
        
        - distribution : C'est une instance de la classe Maillage contenant la distribution de charge.
        
        - maillage : C'est une instance de la classe Maillage contenant le maillage sur lequel on veut calculer le potentiel électrique.
        
        - distribution_charge : C'est un tuple contenant la charge totale et une fonction f qui calcule la distribution de charge.
        NB : Si f est None, la charge est uniformément répartie sur la surface.
        
    Sortie :
            
        - Renvoie un tableau 1D contenant le potentiel électrique calculé sur le maillage donné. 
    """

    k = 1 / (4 * np.pi * cst.epsilon_0)
    val_charge_tot, fonction_charge = distribution_charge

    charge = calcule_charge(distribution, (val_charge_tot, fonction_charge))
    
    r = (-1*distribution.XYZ.T[None, :, :] + maillage.XYZ.T[:, None, :])
    norm : np.ndarray = np.linalg.norm(r,axis=2)
    if filtre:
        norm = np.where(norm >= 1e-6, norm, 1e30)
            
    V = np.sum(k * charge[None,:] / norm,axis=1)

    return V

# **Niveau 6**

In [ ]:
def equipotentielles(potentiel: np.ndarray, maillage_calcule: Maillage, nb_equipotentielles: int = 1) -> list[Maillage]:
    """
    Entrée :
    
        - potentiel : C'est un tableau 1D contenant les valeurs du potentiel électrique sur le maillage.
        
        - maillage_calcule : C'est une instance de la classe Maillage contenant le maillage sur lequel on veut calculer les équipotentielles.
        
        - nb_equipotentielles : C'est un entier positif qui représente le nombre d'équipotentielles à calculer.
        
    Sortie :
    
        - Renvoie une liste de maillages contenant les équipotentielles calculées.
    """
    equipotentielles = []

    if potentiel.ndim != 1:
        raise ValueError("Le tableau de potentiel doit être 1D.")

    seuils = np.linspace(np.nanmin(potentiel), np.nanmax(potentiel), nb_equipotentielles + 1)

    for i in range(nb_equipotentielles):
        vmin = seuils[i]
        vmax = seuils[i + 1]

        if i == nb_equipotentielles - 1:
            indices = np.where((potentiel >= vmin) & (potentiel <= vmax))[0]
        else:
            indices = np.where((potentiel >= vmin) & (potentiel < vmax))[0]

        if len(indices) > 0:
            x = maillage_calcule.X[indices]
            y = maillage_calcule.Y[indices]
            z = maillage_calcule.Z[indices]
            equipotentielles.append((Maillage(x, y, z),np.array([vmin]*len(x))))

    return equipotentielles

In [ ]:
def ligne_de_champ_point(distribution : list[Maillage], point : np.ndarray, distribution_charge : list, nombre_point : int, distance : float = None) -> np.ndarray : 
    """
    Entrée :
    
        - distribution : C'est une liste de maillages contenant la distribution de charge.
        
        - point : C'est un maillage contenant le point de départ de la ligne de champ.
        
        - distribution_charge : C'est une liste de tuples contenant la charge totale et une fonction f qui calcule la distribution de charge.
        NB : Si f est None, la charge est uniformément répartie sur la surface.
        
        - nombre_point : C'est un entier positif qui représente le nombre de points de la ligne de champ.
        
        - distance : C'est un float qui représente la distance entre chaque point de la ligne de champ.
        NB : Si distance est None, la distance entre chaque point est égale à la distance entre le point de départ et le point suivant.
        
    Sortie :
    
        - Renvoie un tableau 2D contenant les coordonnées (X,Y,Z) de la ligne de champ. Les lignes représentent les coordonnées
        et les colonnes représentent les points de la ligne de champ. 
    """
    v : Maillage = (sum(*[calcule_champ(D, Maillage(*point), DC) for D,DC in zip(distribution,distribution_charge)]))[0]
    if distance != None: v /= np.linalg.norm(v)
    liste_point = [point]
        
    for _ in range(1,nombre_point) : 
        point = point + v*distance if distance != None else point + v
        v : Maillage = (sum(*[calcule_champ(D, Maillage(*point), DC) for D,DC in zip(distribution,distribution_charge)]))[0]
        if distance != None and np.linalg.norm(v) >= 1e-6 : v /= np.linalg.norm(v)
        liste_point.append(point)
        
    return np.vstack(liste_point).T

In [ ]:
def non_colineaire(vecteur : np.ndarray) -> np.ndarray :
    """
    Entrée :
    
        - vecteur : C'est un vecteur 3D (1x3) qui représente le vecteur à vérifier.
        NB : Le vecteur doit être non nul.
        
    Sortie :
    
        - Renvoie un vecteur 3D (1x3) qui est colinéaire au vecteur donné.
        NB : Le vecteur renvoyé est le vecteur unitaire dans la direction du vecteur donné.
    """ 
    if vecteur[0] != 0 : 
        return np.array([0,0,vecteur[0]])
    elif vecteur[1] != 0 : 
        return np.array([0,0,vecteur[1]])
    elif vecteur[2] != 0 : 
        return np.array([0,vecteur[2],0])
    
    raise ValueError("Le vecteur doit être non nul.") 

In [ ]:
def decoupe_plan(distribution : Maillage, plan : tuple[np.ndarray,np.ndarray], distance : float) -> np.ndarray : 
    """
    Entrée :
    
        - distribution : C'est une instance de la classe Maillage contenant la distribution de charge.
        
        - plan : C'est un tuple contenant un vecteur normal au plan et un point du plan.
        
        - distance : C'est un float qui représente la distance entre le plan et le maillage.
        NB : Si distance est None, la distance entre le plan et le maillage est égale à la distance entre le point de départ et le point suivant.
    
    Sortie :
    
        - Renvoie un maillage contenant les points de la distribution qui sont à une distance inférieure ou égale à la distance donnée du plan.
        NB : La distance est calculée en utilisant le produit scalaire entre le vecteur normal et le vecteur de position du point.
    """
    if plan[0].any() == 0 : raise ValueError("On ne peut pas prendre le vecteur nul pour définir la normal à un plan.") 
    
    N = plan[0]/np.linalg.norm(plan[0])
    dist = np.dot(distribution.XYZ.T - plan[1],N)
    return np.abs(dist) <= distance 


# **Niveau 7**

In [ ]:
def decoupe_champ(maillage_calcule : Maillage, champ : np.ndarray, plan : tuple[np.ndarray,np.ndarray], distance : float) :
    """
    Entrée :

        - maillage_calcule : C'est une instance de la classe Maillage contenant les points sur lesquels le champ a été évalué.

        - champ : C'est un tableau 2D représentant un champ vectoriel. Les lignes correspondent aux composantes (X, Y, Z),
          et les colonnes aux points du maillage.

        - plan : C'est un tuple de deux éléments :
            * Un vecteur normal au plan (np.ndarray de taille 3).
            * Un point appartenant au plan (np.ndarray de taille 3).

        - distance : C'est un float représentant l'épaisseur autour du plan dans laquelle les points du maillage sont conservés.

    Sortie :

        - Renvoie trois tableaux :

            * start : Un tableau 2D de forme (n, 3) contenant les coordonnées des points sélectionnés, projetés dans une base adaptée au plan.
                      Les colonnes représentent les composantes selon la normale, puis deux directions tangentielles au plan.

            * profondeur : Un tableau 1D de taille n représentant la composante normale du champ projeté sur le plan.
                           C’est une mesure de la "profondeur" du champ à travers le plan.

            * projection_plan : Un tableau 2D de forme (2, n) contenant les composantes du champ projeté tangentes au plan.
                                Ce sont ces composantes qui seront utilisées pour la visualisation du champ projeté.
    """
    P = decoupe_plan(maillage_calcule, plan, distance)
    
    N = plan[0]/np.linalg.norm(plan[0])
    T = np.cross(plan[0],non_colineaire(plan[0]))
    T = T/np.linalg.norm(T)
    B = np.cross(N,T)
    
    Mat_P_C = np.array([N,T,B]).T
    
    x , y, z = np.dot(champ[P],Mat_P_C).T
    projection_plan = np.array([y,z]) 
    start = np.dot(np.array(maillage_calcule[P]).T, Mat_P_C)
    profondeur = x
    return start, profondeur, projection_plan

In [ ]:
def ligne_de_champ(distribution : list[Maillage], maillage_calcule : Maillage, distribution_charge : list, nombre_point : int, distance : float = None) -> np.ndarray :
    """
    Entrée :
    
        - distribution : C'est une liste de maillages contenant la distribution de charge.
        
        - maillage_calcule : C'est un maillage contenant les points de calcule.
        
        - distribution_charge : C'est une liste de tuples contenant la charge totale et une fonction f qui calcule la distribution de charge.
        NB : Si f est None, la charge est uniformément répartie sur la surface.
        
        - nombre_point : C'est un entier positif qui représente le nombre de points de la ligne de champ.
        
        - distance : C'est un float qui représente la distance entre chaque point de la ligne de champ.
        NB : Si distance est None, la distance entre chaque point est égale à la distance entre le point de départ et le point suivant.
        
    Sortie :
    
        - Renvoie un maillage contenant les coordonnées (X,Y,Z) de la ligne de champ. 
    """
    x,y,z = np.concatenate([ligne_de_champ_point(distribution,point,distribution_charge,nombre_point,distance) for point in maillage_calcule],axis=1)
    ligne = Maillage(x,y,z)
    return ligne

In [ ]:
def afficher_plotly(distributions: list[Maillage], 
                    maillages_calcule: list[Maillage] = None, 
                    champs_vectoriels: list[np.ndarray] = None, 
                    taille_distribution: float = 2, 
                    taille_maillage: float = 2, 
                    attributs_distribution: list[tuple[np.ndarray, str]] = None,
                    opacite_distribution = 1
                    ) -> None:
    """
    Entrée :
    
        - distributions : liste de maillages contenant les distributions à afficher.
        
        - maillages_calcule : liste de maillages contenant les maillages calculés.
        
        - champs_vectoriels : liste de champs vectoriels à afficher. Associés dans l'ordre aux maillages calculés.
        
        - taille_distribution : taille des points représentant les distributions.
        
        - taille_maillage : taille des points représentant les maillages.
        
        - attributs_distribution : liste de tuples contenant les attributs des distributions. Chaque tuple contient un tableau de couleurs et un titre. Associés dans l'ordre aux distributions.
        NB : Si attributs_distribution est None, la couleur par défaut est 'cornflowerblue' et le titre est 'Distribution {idx + 1}'.
        
    Sortie :
    
        - Renvoie None. Affiche le graphique 3D interactif.
    """
    fig = go.Figure()
    
    if attributs_distribution is None:
        attributs_distribution = [(None, None)] * len(distributions)
    elif len(attributs_distribution) != len(distributions):
        raise ValueError("Il doit y avoir autant d'attributs que de distributions. Mettre (None, None) si pas d'attributs.")

    palette = ['Inferno', 'Viridis', 'Cividis', 'Plasma', 'Magma', 'Turbo']
    
    for idx, (distribution, attribut) in enumerate(zip(distributions, attributs_distribution)):
        couleur, titre = attribut
        colorbar = None
        if couleur is None:
            scatter_color = dict(size=taille_distribution, color='cornflowerblue', opacity=opacite_distribution)
        if isinstance(couleur, str):
            scatter_color = dict(size=taille_distribution, color=couleur, opacity=opacite_distribution)
        elif isinstance(couleur, np.ndarray) and titre:
            colorbar = dict(title=titre, len=0.8, x=1.05 - 0.1 * idx, y=0.4, tickformat=".2e")
            scatter_color = dict(size=taille_distribution, color=couleur, colorscale=palette[idx % len(palette)],
                                 colorbar=colorbar, opacity=opacite_distribution, cmin = np.nanmin(couleur), cmax = np.nanmax(couleur))
        
        fig.add_trace(go.Scatter3d(
            x=distribution.X, y=distribution.Y, z=distribution.Z,
            mode='markers',
            marker=scatter_color,
            name=titre if titre else f'Distribution {idx + 1}',
            showlegend=True
        ))

    if champs_vectoriels is None:
        fig.show()
        return None
    
    if maillages_calcule is None:
        fig.show()
        return None
    
    elif len(maillages_calcule) != len(champs_vectoriels) and champs_vectoriels is not None:
        raise ValueError("Le nombre de maillages et de champs vectoriels doit être le même.")
    
    for maillage_calcule, champ_vectoriel in zip(maillages_calcule, champs_vectoriels):
        for i in range(maillage_calcule.XYZ.shape[1]):
            x0, y0, z0 = maillage_calcule.X[i], maillage_calcule.Y[i], maillage_calcule.Z[i]
            dx, dy, dz = champ_vectoriel[i]
            norme = np.linalg.norm(champ_vectoriel[i])

            fig.add_trace(go.Scatter3d(
                x=[x0, x0 + dx], y=[y0, y0 + dy], z=[z0, z0 + dz],
                mode='lines',
                line=dict(color='red', width=2),
                showlegend=False,
                name='Champ vectoriel',
                hoverinfo='text',
                text=f"norme = {norme:.6f}"
            ))

            fig.add_trace(go.Scatter3d(
                x=[x0 + dx], y=[y0 + dy], z=[z0 + dz],
                mode='markers',
                marker=dict(size=taille_maillage, color='red', opacity=0.6),
                name='Champ vectoriel',
                text=f"norme = {norme:.6f}",
                hoverinfo='text',                
                showlegend=False
            ))

    fig.update_layout(
        title="Représentation du champ vectoriel",
        template='plotly_white',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()
    return None


# **Niveau 8**

In [ ]:
def projection_champ_plotly(maillage_calcule : Maillage, champ : np.ndarray, plan : tuple[np.ndarray,np.ndarray], distance : float) -> None : 
    """
    Entrée :

        - maillage_calcule : C'est une instance de la classe Maillage contenant les points sur lesquels le champ a été évalué.

        - champ : C'est un tableau 2D représentant un champ vectoriel. Les lignes correspondent aux composantes (X, Y, Z),
          et les colonnes aux points du maillage.

        - plan : C'est un tuple de deux éléments :
            * Un vecteur directeur normal au plan (np.ndarray de taille 3).
            * Un point appartenant au plan (np.ndarray de taille 3).

        - distance : C'est un float représentant l'épaisseur autour du plan dans laquelle les points du maillage sont conservés.

    Sortie :

        - Aucun objet n'est retourné. La fonction affiche une figure Plotly contenant :
            * Une projection du champ vectoriel sur le plan donné.
            * Les vecteurs représentés comme des flèches ancrées aux projections des points du maillage.
            * Les couleurs des flèches codent la composante normale du champ (profondeur).
        
        NB : Si aucun point du maillage ne se trouve dans l'épaisseur définie par `distance`, une erreur est levée.
    """
    start, profondeur, projection = decoupe_champ(maillage_calcule,champ, plan ,distance)
    _, x, y = start.T
    vx,vy = projection

    fig = go.Figure()

    if len(profondeur) == 0:
        raise ValueError("Aucun point dans le plan.")
        
    colors = px.colors.sample_colorscale('viridis', (profondeur - profondeur.min()) / (profondeur.max() - profondeur.min()))

    for i in range(len(y)):
        fig.add_trace(go.Scatter(
            x=[x[i], x[i]+vx[i]],
            y=[y[i], y[i]+vy[i]],
            mode='lines',
            line=dict(color=colors[i]),
            showlegend=False
        ))

    fig.add_trace(go.Scatter(
        x=x+vx,
        y=y+vy,
        mode='markers',
        marker=dict(
            size=8,
            color= profondeur,
            colorscale='viridis',
            opacity=1,
            colorbar=dict(title='Valeur')
        ),
        showlegend=False
    ))

    fig.show()
    return None

# **Fonctionnalités**

## **Plan**

In [ ]:
cote = 4

plan = creer_distribution(parametrage_plan,(-cote/2,cote/2),(-cote/2,cote/2), 100)
plan.afficher_3D_plotly()

In [ ]:
aire = plan.calculer_aire()
print(f"L'aire du plan est de {aire:.3f}")

In [ ]:
dS = plan.surface[0].calculer_dS()
afficher_plotly([plan],attributs_distribution=[(dS,"dS ($unite^2$)")])

In [ ]:
charge = 1e-9
dq_uniforme = calcule_charge(plan,(charge,None))
print(f'La valeur de la charge totale est de {np.sum(dq_uniforme):.10e}C.')
afficher_plotly([plan],attributs_distribution=[(dq_uniforme,"dq (C)")])

In [ ]:
dq_non_uniforme = calcule_charge(plan,(charge, lambda u,v : u**2+v**2))
print(f'La valeur de la charge totale est de {np.sum(dq_uniforme):.10e}C.')
afficher_plotly([plan],attributs_distribution=[(dq_non_uniforme,"dq (C)")])

In [ ]:
maillage_calcule = sum(*[creer_distribution(parametrage_plan, (-cote,cote),(-cote,cote), 5, i) for i in np.arange(-3,0)])
maillage_calcule += sum(*[creer_distribution(parametrage_plan, (-cote,cote),(-cote,cote), 5, i) for i in np.arange(1,4)])
maillage_calcule += creer_distribution(parametrage_plan, (cote,cote),(-cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,-cote),(-cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(-cote,-cote), 5, 0)
champ_electrique = calcule_champ(plan, maillage_calcule, (charge, None))

afficher_plotly([plan],[maillage_calcule],[champ_electrique])

In [ ]:
champ_electrique = calcule_champ(plan, maillage_calcule, (charge, lambda u,v : u**2 + v**2))

afficher_plotly([plan],[maillage_calcule],[champ_electrique], attributs_distribution=[(dq_non_uniforme,'dq (C)')])

In [ ]:
potentiel = calcule_potentiel(plan, maillage_calcule, (charge, None))
afficher_plotly([plan, maillage_calcule],attributs_distribution=[(None,None),(potentiel,"potentiel (V)")])

In [ ]:
potentiel = calcule_potentiel(plan, maillage_calcule, (charge, lambda u,v : u**2 + v**2))
afficher_plotly([plan, maillage_calcule],attributs_distribution=[(dq_non_uniforme,'dq (C)'),(potentiel,"potentiel (V)")])

In [ ]:
maillage_ligne1 = creer_distribution(parametrage_plan, (-cote,cote), (-cote,cote), 5, 1)
maillage_ligne2 = creer_distribution(parametrage_plan, (-cote,cote), (-cote,cote), 5,-1)
ligne1 = ligne_de_champ([plan],maillage_ligne1, [(charge, None)], 20, 1)
ligne2 = ligne_de_champ([plan],maillage_ligne2, [(charge, None)], 20, 1)


afficher_plotly([plan, ligne1, ligne2], attributs_distribution=[(None,None),("red",None),("red",None)], taille_distribution=1)

In [ ]:
ligne1_bis = ligne_de_champ([plan],maillage_ligne1, [(charge, lambda u,v : u**2 + v**2)], 20, 1)
ligne2_bis = ligne_de_champ([plan],maillage_ligne2, [(charge, lambda u,v : u**2 + v**2)], 20, 1)


afficher_plotly([plan, ligne1_bis, ligne2_bis], attributs_distribution=[(dq_non_uniforme,"dq"),("red",None),("red",None)])

In [ ]:
distance_plaque = 2

plaque_positive = creer_distribution(parametrage_plan,(-cote/2,cote/2),(-cote/2,cote/2), 100, -distance_plaque/2)
plaque_negative = creer_distribution(parametrage_plan,(-cote/2,cote/2),(-cote/2,cote/2), 100, distance_plaque/2)

dq_uniforme_plaque_positive = calcule_charge(plaque_positive, (charge,None))
dq_uniforme_plaque_negative = calcule_charge(plaque_positive, (-charge,None))

afficher_plotly([plaque_positive,plaque_negative],attributs_distribution=[(dq_uniforme_plaque_positive,"dq+ (C)"),(dq_uniforme_plaque_negative,"dq- (C)")])

In [ ]:
maillage_calcule = sum(*[creer_distribution(parametrage_plan, (-cote,cote),(-cote,cote), 5, i) for i in np.arange(-distance_plaque - 2,-distance_plaque)])
maillage_calcule += sum(*[creer_distribution(parametrage_plan, (-cote,cote),(-cote,cote), 5, i) for i in np.arange(distance_plaque + 1,distance_plaque + 3)])
maillage_calcule += sum(*[creer_distribution(parametrage_plan, (-cote,cote),(-cote,cote), 5, i) for i in np.arange(-distance_plaque/2 + 1,distance_plaque/2)])
maillage_calcule += creer_distribution(parametrage_plan, (cote,cote),(-cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,-cote),(-cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(-cote,-cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (cote,cote),(-cote,cote), 5, 1)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,-cote),(-cote,cote), 5, 1)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(cote,cote), 5, 1)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(-cote,-cote), 5, 1)
maillage_calcule += creer_distribution(parametrage_plan, (cote,cote),(-cote,cote), 5, -1)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,-cote),(-cote,cote), 5, -1)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(cote,cote), 5, -1)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(-cote,-cote), 5, -1)


champ_electrique_negative = calcule_champ(plaque_negative,maillage_calcule,(-charge,None))
champ_electrique_positive = calcule_champ(plaque_positive,maillage_calcule,(charge,None))

afficher_plotly([plaque_positive,plaque_negative],[maillage_calcule],[champ_electrique_negative + champ_electrique_positive],attributs_distribution=[(dq_uniforme_plaque_positive,"dq+ (C)"),(dq_uniforme_plaque_negative,"dq- (C)")])

In [ ]:
potentiel_positif = calcule_potentiel(plaque_positive,maillage_calcule,(charge,None))
potentiel_negatif = calcule_potentiel(plaque_negative,maillage_calcule,(-charge,None))

afficher_plotly([plaque_positive, plaque_negative, maillage_calcule],attributs_distribution=[(dq_uniforme_plaque_positive,"dq+"),
                                                                                                 (dq_uniforme_plaque_negative,"dq-"),
                                                                                                 (potentiel_positif + potentiel_negatif,"potentiel (V)")])

In [ ]:
equi = equipotentielles(potentiel_positif + potentiel_negatif,maillage_calcule, 100)
afficher_plotly([plaque_negative,plaque_positive,equi[5][0]], attributs_distribution=[(dq_uniforme_plaque_negative,'dq-'),(dq_uniforme_plaque_positive,'dq+'),(equi[5][1], 'potentiel')])

In [ ]:
maillage_calcule = creer_distribution(parametrage_plan,(-cote/2,cote/2),(-cote/2,cote/2),5,-distance_plaque/2+0.1)
ligne = ligne_de_champ([plaque_negative,plaque_positive],maillage_calcule,[(-charge,None),(charge,None)], 23,0.1)

afficher_plotly([plaque_positive,plaque_negative, ligne], attributs_distribution=[(None,None),(None,None),("red",None)], taille_distribution=2)

In [ ]:
maillage_calcule = creer_distribution(parametrage_disque,(1,1),(0,2*np.pi,True),5,-distance_plaque/2-0.1)
ligne = ligne_de_champ([plaque_negative,plaque_positive],maillage_calcule,[(-charge,None),(charge,None)], 60,5)

afficher_plotly([plaque_positive,plaque_negative, ligne], attributs_distribution=[(None,None),(None,None),("red",None)], taille_distribution=2)

---

## **Le disque**

In [ ]:
disque = creer_distribution(parametrage_disque,(0,1),(0,2*np.pi,True), 100)
disque.afficher_3D_plotly()

In [ ]:
aire = disque.calculer_aire()
print(f"L'aire du plan est de {aire:.3f}")

In [ ]:
dS = disque.surface[0].calculer_dS()
afficher_plotly([disque],attributs_distribution=[(dS,"dS ($unite^2$)")])

In [ ]:
charge = 1e-9
dq_uniforme = calcule_charge(disque,(charge,None))
print(f'La valeur de la charge totale est de {np.sum(dq_uniforme):.10e}C.')
afficher_plotly([disque],attributs_distribution=[(dq_uniforme,"dq (C)")])

In [ ]:
dq_non_uniforme = calcule_charge(disque,(charge, lambda u,v : u**2))
print(f'La valeur de la charge totale est de {np.sum(dq_uniforme):.10e}C.')
afficher_plotly([disque],attributs_distribution=[(dq_non_uniforme,"dq (C)")])

In [ ]:
maillage_calcule = sum(*[creer_distribution(parametrage_plan, (-cote,cote),(-cote,cote), 5, i) for i in np.arange(-3,0)])
maillage_calcule += sum(*[creer_distribution(parametrage_plan, (-cote,cote),(-cote,cote), 5, i) for i in np.arange(1,4)])
maillage_calcule += creer_distribution(parametrage_plan, (cote,cote),(-cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,-cote),(-cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(cote,cote), 5, 0)
maillage_calcule += creer_distribution(parametrage_plan, (-cote,cote),(-cote,-cote), 5, 0)
champ_electrique = calcule_champ(disque, maillage_calcule, (charge, None))

afficher_plotly([disque],[maillage_calcule],[champ_electrique])

In [ ]:
champ_electrique = calcule_champ(disque, maillage_calcule, (charge, lambda u,v : u**2))

afficher_plotly([disque],[maillage_calcule],[champ_electrique], attributs_distribution=[(dq_non_uniforme,'dq (C)')])

In [ ]:
potentiel = calcule_potentiel(disque, maillage_calcule, (charge, None))
afficher_plotly([disque, maillage_calcule],attributs_distribution=[(None,None),(potentiel,"potentiel (V)")])

In [ ]:
potentiel = calcule_potentiel(disque, maillage_calcule, (charge, lambda u,v : u**2))
afficher_plotly([disque, maillage_calcule],attributs_distribution=[(dq_non_uniforme,'dq (C)'),(potentiel,"potentiel (V)")])

--- 

## **Sphère**

In [ ]:
sphere = creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi), 100,1)
sphere.afficher_3D_plotly()

In [ ]:
aire = sphere.calculer_aire()
print(f"L'aire du plan est de {aire:.3f}")

In [ ]:
dS = sphere.surface[0].calculer_dS()
afficher_plotly([sphere],attributs_distribution=[(dS,"dS ($unite^2$)")])

In [ ]:
charge = 1e-10
dq_uniforme = calcule_charge(sphere,(charge,None))
print(f'La valeur de la charge totale est de {np.sum(dq_uniforme):.10e}C.')
afficher_plotly([sphere],attributs_distribution=[(dq_uniforme,"dq (C)")])

In [ ]:
maillages_calcule = creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi), 10,2)
champ_electrique = calcule_champ(sphere, maillages_calcule, (charge, None))

afficher_plotly([sphere],[maillages_calcule],[champ_electrique])

In [ ]:
projection_champ_plotly(maillages_calcule,champ_electrique,(np.array([0,0,1]),np.array([0,2,0])),0.5)

In [ ]:
charge = 1e-15
maillages_calcule = creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi), 10,2)

sigma = lambda u,v : np.cos(v)

dq_non_uniforme = calcule_charge(sphere,(charge, sigma))
champ_electrique = calcule_champ(sphere, maillages_calcule, (charge, sigma))

afficher_plotly([sphere],[maillages_calcule],[champ_electrique], attributs_distribution=[(dq_non_uniforme,'dq (C)')])

In [ ]:
charge  = 1e-10
maillages_calcule = sum(*[creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi), 10,1.5 + 0.5*i) for i in range(3)])

potentiel = calcule_potentiel(sphere, maillages_calcule, (charge, None))
afficher_plotly([sphere, maillages_calcule],attributs_distribution=[(None,None),(potentiel,"potentiel (V)")])

In [ ]:
potentiel = calcule_potentiel(sphere, maillages_calcule, (charge, sigma))
afficher_plotly([sphere, maillages_calcule],attributs_distribution=[(dq_non_uniforme,'dq (C)'),(potentiel,"potentiel (V)")])

In [ ]:
maillage_ligne1 = creer_distribution(parametrage_disque, (0.5,0.5), (0,2*np.pi,True), 5, 0.9)
maillage_ligne2 = creer_distribution(parametrage_disque, (0.5,0.5), (0,2*np.pi,True), 5, -0.9)

ligne1 = ligne_de_champ([sphere],maillage_ligne1, [(charge, None)], 20, 0.1)
ligne2 = ligne_de_champ([sphere],maillage_ligne2, [(charge, None)], 20, 0.1)

afficher_plotly([sphere, ligne1, ligne2], attributs_distribution=[(None,None),("red",None),("red",None)], taille_distribution=1)

In [ ]:
maillage_ligne1 = creer_distribution(parametrage_disque, (0.5,0.5), (0,2*np.pi,True), 5, 0.9)
maillage_ligne2 = creer_distribution(parametrage_disque, (0.5,0.5), (0,2*np.pi,True), 5, -0.9)

ligne1 = ligne_de_champ([sphere],maillage_ligne1, [(charge, sigma)], 30, 0.5)

afficher_plotly([sphere, ligne1], attributs_distribution=[(dq_non_uniforme,'dq'),("red",None)], taille_distribution=1)

In [ ]:
sphere_positive = creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi),100,1)
sphere_negative = creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi),100,2)

dq_positive = calcule_charge(sphere_positive, (charge, None))
dq_negative = calcule_charge(sphere_negative, (-charge, None))

afficher_plotly([sphere_negative,sphere_positive], attributs_distribution=[(dq_negative,'dq-'),(dq_positive,'dq+')])

In [ ]:
charge = 1e-10

maillages_calcule1 = creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi), 10,0.5)
maillages_calcule2 = sum(*[creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi), 10,2.5 + 0.5*i) for i in range(3)])
maillages_calcule3 = creer_distribution(parametrage_sphere,(0,2*np.pi,True),(0,np.pi), 10,1.5)

champ_electrique_p1 = calcule_champ(sphere_positive,maillages_calcule1,(charge,None))
champ_electrique_n1 = calcule_champ(sphere_negative,maillages_calcule1,(-charge,None))

champ_electrique_p2 = calcule_champ(sphere_positive,maillages_calcule2,(charge,None))
champ_electrique_n2 = calcule_champ(sphere_negative,maillages_calcule2,(-charge,None))

champ_electrique_p3 = calcule_champ(sphere_positive,maillages_calcule3,(charge,None))
champ_electrique_n3 = calcule_champ(sphere_negative,maillages_calcule3,(-charge,None))

#afficher_plotly([sphere_negative,sphere_positive],[maillages_calcule1, maillages_calcule2, maillages_calcule3],[champ_electrique_n1+champ_electrique_p1, champ_electrique_n2+champ_electrique_p2, champ_electrique_p3+champ_electrique_n3])
#afficher_plotly([sphere_negative,sphere_positive],[maillages_calcule1],[champ_electrique_n1+champ_electrique_p1])
#afficher_plotly([sphere_negative,sphere_positive],[maillages_calcule2],[champ_electrique_n2+champ_electrique_p2])
afficher_plotly([sphere_negative,sphere_positive],[maillages_calcule3],[champ_electrique_p3+champ_electrique_n3], attributs_distribution=[(dq_negative, 'dq+'),(dq_positive,'dq-')])


In [ ]:
potentiel_p1 = calcule_potentiel(sphere_positive,maillages_calcule1,(charge,None))
potentiel_n1 = calcule_potentiel(sphere_negative,maillages_calcule1,(-charge,None))

potentiel_p2 = calcule_potentiel(sphere_positive,maillages_calcule2,(charge,None))
potentiel_n2 = calcule_potentiel(sphere_negative,maillages_calcule2,(-charge,None))

potentiel_p3 = calcule_potentiel(sphere_positive,maillages_calcule3,(charge,None))
potentiel_n3 = calcule_potentiel(sphere_negative,maillages_calcule3,(-charge,None))

#afficher_plotly([sphere_negative,sphere_positive, maillages_calcule1, maillages_calcule2, maillages_calcule3], attributs_distribution=[(None,None),(None,None),(potentiel_n1+potentiel_p1,'potentiel'),(potentiel_n2+potentiel_p2,'potentiel'),(potentiel_n3+potentiel_p3,'potentiel')])
#afficher_plotly([sphere_negative,sphere_positive,maillages_calcule1],attributs_distribution=[(None,None),(None,None),(potentiel_n1+potentiel_p1,'potentiel')])
#afficher_plotly([sphere_negative,sphere_positive,maillages_calcule2],attributs_distribution=[(None,None),(None,None),(potentiel_n2+potentiel_p2,'potentiel')])
afficher_plotly([sphere_negative,sphere_positive,maillages_calcule3],attributs_distribution=[(None,None),(None,None),(potentiel_n3+potentiel_p3,'potentiel')])

---

## **Le cylindre**

In [ ]:
cylindre = creer_distribution(parametrage_disque,(0,1),(0,2*np.pi,True),100,0) + creer_distribution(parametrage_disque,(0,1),(0,2*np.pi,True),100,4) + creer_distribution(parametrage_cylindre,(0,2*np.pi,True),(0,4),100)
cylindre.afficher_3D_plotly()

In [ ]:
aire = cylindre.calculer_aire()
print(f"L'aire du plan est de {aire:.3f}")

In [ ]:
dq = calcule_charge(cylindre, (charge,None))
afficher_plotly([cylindre],attributs_distribution=[(dq,'dq')])

In [ ]:
charge = 1e-9
maillage_calcule = sum(*[creer_distribution(parametrage_cylindre, (0,2*np.pi,True),(-2,6), 10, i + 2) for i in range(5)])
champ = calcule_champ(cylindre, maillage_calcule, (charge, None))

afficher_plotly([cylindre],[maillage_calcule],[champ])

In [ ]:
charge = 1e-9
maillage_calcule = sum(*[creer_distribution(parametrage_cylindre, (0,2*np.pi,True),(-2,6), 10, i + 2) for i in range(5)])
potentiel = calcule_potentiel(cylindre, maillage_calcule, (charge, None))

afficher_plotly([cylindre, maillage_calcule],attributs_distribution=[(None,None),(potentiel,'potentiel')])

In [ ]:
charge = 4e-11

cylindre_interiueur = creer_distribution(parametrage_disque,(0,0.5),(0,2*np.pi,True),100,0) + creer_distribution(parametrage_disque,(0,0.5),(0,2*np.pi,True),100,4) + creer_distribution(parametrage_cylindre,(0,2*np.pi,True),(0.5,3.5),100,0.5)
maillage_calcule = creer_distribution(parametrage_disque,(0,0.75),(0,2*np.pi,True),10,0.25) + creer_distribution(parametrage_disque,(0,0.75),(0,2*np.pi,True),10,3.75) + creer_distribution(parametrage_cylindre,(0,2*np.pi,True),(0.25,3.75),10,0.75)

champ_p= calcule_champ(cylindre_interiueur,maillage_calcule,(charge,None))
champ_n= calcule_champ(cylindre,maillage_calcule,(-charge,None))
afficher_plotly([cylindre,cylindre_interiueur],[maillage_calcule],[champ_p+champ_n])

---

## **Résultats tiers**

In [ ]:
tore = creer_distribution(parametrage_tore,(0,2*np.pi,True),(0,2*np.pi,True),100,3,1)
maillage_calcule = creer_distribution(parametrage_tore,(0,2*np.pi,True),(0,2*np.pi,True),10,3,1.5)

champ = calcule_champ(tore,maillage_calcule,(1e-9,None))

afficher_plotly([tore],[maillage_calcule],[champ])

In [ ]:
projection_champ_plotly(maillage_calcule,champ,(np.array([0,1,0]),np.array([0,0,0])),0.1)

In [ ]:
angle = np.pi/4
hauteur = 0.5
L = 6
etage = int(np.sin(angle)*L/hauteur) + 1 


plan =  creer_distribution(parametrage_plan,(0,L),(0,L),100,-0.5) + creer_distribution(parametrage_plan,(0,L),(0,L),100,0.5).rotate(['x'],[angle])
maillage_calcule = sum(*[creer_distribution(parametrage_plan,(0,L),(np.cos(angle)*hauteur*i/np.sin(angle),L),5,hauteur*i) for i in range(etage)])
champ = calcule_champ(plan,maillage_calcule,(1e-9,None))
afficher_plotly([plan],[maillage_calcule],[champ])

In [ ]:
projection_champ_plotly(maillage_calcule,champ,(np.array([-1,0,0]),np.array([0,0,100])),0.2)